In [27]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("D:/Dev/enterprise-aws-data-lakehouse-ml-system")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

train = pd.read_parquet(DATA_PROCESSED / "train_foundation.parquet")

print(train.shape)

(590540, 440)


In [28]:
train

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_37,id_38,DeviceType,DeviceInfo,day,hour,card1_freq,card2_freq,card3_freq,card4_freq
0,2987000,0,86400,68.50,W,13926,NaN,150.0,discover,142.0,...,None,None,None,None,1,0,43,8933,521287,6651
1,2987001,0,86401,29.00,W,2755,404.0,150.0,mastercard,102.0,...,None,None,None,None,1,0,683,3056,521287,189217
2,2987002,0,86469,59.00,W,4663,490.0,150.0,visa,166.0,...,None,None,None,None,1,0,1108,38145,521287,384767
3,2987003,0,86499,50.00,W,18132,567.0,150.0,mastercard,117.0,...,None,None,None,None,1,0,4209,6137,521287,189217
4,2987004,0,86506,50.00,H,4497,514.0,150.0,mastercard,102.0,...,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M,1,0,18,14541,521287,189217
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,None,None,None,None,182,23,1183,8933,521287,384767
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,None,None,None,None,182,23,12,7445,521287,189217
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,None,None,None,None,182,23,690,734,521287,189217
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,None,None,None,None,182,23,3006,6336,521287,189217


In [29]:
"addr1" in train

True

### Create UID v1

In [30]:
train["uid"] = (
    train["card1"].astype(str) + "_" +
    train["addr1"].astype(str)
)

### Sort by UID + Time

In [31]:
train = train.sort_values(["uid", "TransactionDT"])

In [32]:
train

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_38,DeviceType,DeviceInfo,day,hour,card1_freq,card2_freq,card3_freq,card4_freq,uid
182988,3169988,0,4050851,29.000,W,10000,111.0,150.0,mastercard,117.0,...,None,None,None,46,21,1,45191,521287,189217,10000_184.0
314550,3301550,0,7840676,42.777,C,10003,NaN,NaN,None,NaN,...,F,mobile,S60 Build/MMB29M,90,17,5,8933,1565,1577,10003_nan
341484,3328484,0,8421815,39.394,C,10003,555.0,128.0,visa,226.0,...,F,mobile,S60 Build/MMB29M,97,11,5,41995,7,384767,10003_nan
350343,3337343,0,8634882,10.755,C,10003,555.0,128.0,visa,226.0,...,F,mobile,S60 Build/MMB29M,99,22,5,41995,7,384767,10003_nan
350365,3337365,0,8635215,19.093,C,10003,555.0,128.0,visa,226.0,...,F,mobile,S60 Build/MMB29M,99,22,5,41995,7,384767,10003_nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533757,3520757,0,14063430,25.000,S,9999,174.0,150.0,visa,226.0,...,T,desktop,rv:11.0,162,18,18,11310,521287,384767,9999_330.0
542705,3529705,0,14320277,20.000,S,9999,174.0,150.0,visa,226.0,...,T,desktop,rv:11.0,165,17,18,11310,521287,384767,9999_330.0
545261,3532261,0,14398268,20.000,S,9999,174.0,150.0,visa,226.0,...,T,desktop,rv:11.0,166,15,18,11310,521287,384767,9999_330.0
548099,3535099,0,14482385,30.000,S,9999,174.0,150.0,visa,226.0,...,T,desktop,rv:11.0,167,14,18,11310,521287,384767,9999_330.0


In [ ]:
train["uid_next_dt"] = train.groupby("uid")["TransactionDT"].shift(-1)  # shift() moves values up or down within each group.
train["uid_prev_dt"] = train.groupby("uid")["TransactionDT"].shift(1)

train["uid_time_to_next"] = train["uid_next_dt"] - train["TransactionDT"]
train["uid_time_from_prev"] = train["TransactionDT"] - train["uid_prev_dt"]

### UID Transaction Count

In [34]:
train["uid_txn_count"] = train.groupby("uid")["TransactionID"].transform("count")

### UID Statistical Aggregations

In [35]:
train["uid_amt_mean"] = train.groupby("uid")["TransactionAmt"].transform("mean")
train["uid_amt_std"] = train.groupby("uid")["TransactionAmt"].transform("std")
train["uid_amt_median"] = train.groupby("uid")["TransactionAmt"].transform("median")

### Contextual Deviation Feature

In [ ]:
train["uid_amt_deviation"] = (
    train["TransactionAmt"] - train["uid_amt_mean"]
) / (train["uid_amt_std"] + 1e-6)  # Identity-aware contextual deviation of transaction amount
                                   # If uid_amt_std = 0 answer will infinity. It isn't suitable

In [37]:
train

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,uid,uid_next_dt,uid_prev_dt,uid_time_to_next,uid_time_from_prev,uid_txn_count,uid_amt_mean,uid_amt_std,uid_amt_median,uid_amt_deviation
182988,3169988,0,4050851,29.000,W,10000,111.0,150.0,mastercard,117.0,...,10000_184.0,NaN,NaN,NaN,NaN,1,29.000000,NaN,29.000,NaN
314550,3301550,0,7840676,42.777,C,10003,NaN,NaN,None,NaN,...,10003_nan,8421815.0,NaN,581139.0,NaN,5,26.222400,14.039613,19.093,1.179135
341484,3328484,0,8421815,39.394,C,10003,555.0,128.0,visa,226.0,...,10003_nan,8634882.0,7840676.0,213067.0,581139.0,5,26.222400,14.039613,19.093,0.938174
350343,3337343,0,8634882,10.755,C,10003,555.0,128.0,visa,226.0,...,10003_nan,8635215.0,8421815.0,333.0,213067.0,5,26.222400,14.039613,19.093,-1.101697
350365,3337365,0,8635215,19.093,C,10003,555.0,128.0,visa,226.0,...,10003_nan,8642405.0,8634882.0,7190.0,333.0,5,26.222400,14.039613,19.093,-0.507806
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533757,3520757,0,14063430,25.000,S,9999,174.0,150.0,visa,226.0,...,9999_330.0,14320277.0,13185125.0,256847.0,878305.0,17,23.529412,7.657657,20.000,0.192042
542705,3529705,0,14320277,20.000,S,9999,174.0,150.0,visa,226.0,...,9999_330.0,14398268.0,14063430.0,77991.0,256847.0,17,23.529412,7.657657,20.000,-0.460900
545261,3532261,0,14398268,20.000,S,9999,174.0,150.0,visa,226.0,...,9999_330.0,14482385.0,14320277.0,84117.0,77991.0,17,23.529412,7.657657,20.000,-0.460900
548099,3535099,0,14482385,30.000,S,9999,174.0,150.0,visa,226.0,...,9999_330.0,NaN,14398268.0,NaN,84117.0,17,23.529412,7.657657,20.000,0.844983


In [38]:
train.to_parquet(
    DATA_PROCESSED / "train_behavioral.parquet",
    index=False
)